# clue_policy

> Phase 12: prompts rendered exactly as Phase 10's `LLMMember` renders them, reply parsing, and policies (functions from a batch of prompts to replies): scripted and Claude. Local models are in `local_policy`.

In [ ]:
#| default_exp clue_policy

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import json
from concurrent.futures import ThreadPoolExecutor

from explRL.clue_text import CLUE_SCHEMA
from explRL.llm_agent import FRAMING
from explRL.llm_client import LLMReply

USAGE0 = dict(input_tokens=0, cache_creation_input_tokens=0, cache_read_input_tokens=0, output_tokens=0)


def historyBlocks(memory='notes', notes='', past=()):
    "Past turns as LLMMember renders them: the latest note ('notes') or every summary line ('full')."
    if memory == 'notes':
        texts = [f'Your notes so far: {notes}'] if notes else []
    elif memory == 'full':
        texts = ['Your past turns, oldest first:\n' + past[0]] + list(past[1:]) if past else []
    else:
        raise ValueError(f'unknown memory {memory}')
    blocks = [{'type': 'text', 'text': x} for x in texts]
    if blocks:
        blocks[-1]['cache_control'] = {'type': 'ephemeral'}
    return blocks


def renderPrompt(env, obs, memory='notes', notes='', past=(), framing='own'):
    "(system, blocks) exactly as LLMMember sends them for this state."
    system = env.rules() + '\n\n' + FRAMING[framing].format(u=env.unit)
    return system, historyBlocks(memory, notes, past) + [{'type': 'text', 'text': 'Now:\n' + obs}]


def userText(blocks):
    "The blocks as one user message (local models: a system message and one user message)."
    return '\n\n'.join(b['text'] for b in blocks)


def parseReply(text):
    "The reply's JSON object: the whole text, else the first decodable {...} in it; None if there is none."
    if not isinstance(text, str):
        return None
    try:
        x = json.loads(text)
        return x if isinstance(x, dict) else None
    except ValueError:
        pass
    dec = json.JSONDecoder()
    for i, c in enumerate(text):
        if c == '{':
            try:
                x, _ = dec.raw_decode(text, i)
            except ValueError:
                continue
            if isinstance(x, dict):
                return x
    return None


def asReply(text, usage=None, stop='end_turn'):
    "A local or scripted reply as an LLMReply (cost 0)."
    return LLMReply(text, parseReply(text), stop, None, dict(USAGE0, **(usage or {})), 0.0)


def scriptedPolicy(fn):
    "A policy from fn(system, blocks) -> reply text (tests, replays)."
    return lambda requests: [asReply(fn(s, b)) for s, b in requests]


def claudePolicy(client, workers=8, schema=CLUE_SCHEMA, task='clue'):
    """A policy that sends each request through a Phase 10 LLMClient with `schema` (`task`: the timing records' task;
    Phase 13: the clue board's options and the vote pass theirs); a batch's requests concurrently."""
    def policy(requests):
        with ThreadPoolExecutor(max(1, min(workers, len(requests)))) as ex:
            return list(ex.map(lambda r: client.complete(r[0], r[1], schema, task=task), requests))
    return policy

## Tests

In [ ]:
import json
from explRL.clue_policy import *
from explRL.clue_agents import ClueBoard
from explRL.clue_text import CLUE_SCHEMA, TextClueEnv
from explRL.clueboard import ClueBoardEnv
from explRL.llm_agent import LLMMember
from explRL.llm_client import LLMClient
from explRL.llm_fake import FakeClient, fakeReply

# renderPrompt equals what LLMMember sends, for notes and full memory, framings and with or without history
b = ClueBoardEnv(1, 4, m=2, r=1, R=30); env = TextClueEnv(b); ch = ClueBoard()
for memory in ('notes', 'full'):
    for framing in ('own', 'team'):
        m = LLMMember(None, env, ch, memory, framing); m.id = 2
        for notes, past in (('', []), ('try T3', ['turn 0: opened T3, earned 0.00.', 'turn 1: waited, earned 0.00.'])):
            m.notes, m.past = notes, list(past)
            obs, system, blocks = m.prepare(b, 1.0)
            assert renderPrompt(env, obs, memory, notes, past, framing) == (system, blocks), (memory, framing, notes)
# parseReply: whole JSON, embedded JSON, junk
assert userText([{'type': 'text', 'text': 'a'}, {'type': 'text', 'text': 'Now:\nb'}]) == 'a\n\nNow:\nb'
assert parseReply('{"action": "wait"}') == {'action': 'wait'}
assert parseReply('Sure! {"action": "open", "target": "T1"} done') == {'action': 'open', 'target': 'T1'}
assert parseReply('[1, 2]') is None and parseReply('no json {here') is None and parseReply(None) is None
r = asReply('{"notes": "x", "action": "wait"}', usage=dict(output_tokens=5))
assert r.parsed['action'] == 'wait' and r.usage['output_tokens'] == 5 and r.cost == 0 and r.stopReason == 'end_turn'
# scripted and Claude (fake client) policies return one reply per request, in order
pol = scriptedPolicy(lambda s, bl: json.dumps(dict(action='wait', n=len(bl))))
out = pol([('s', [1]), ('s', [1, 2])])
assert [x.parsed['n'] for x in out] == [1, 2]
client = LLMClient('claude-haiku-5-5', client=FakeClient(fakeReply))
reqs = [renderPrompt(env, env.observe(i, 2.0, ch)) for i in range(4)]
seq = [client.complete(s, bl, CLUE_SCHEMA, task='clue').text for s, bl in reqs]
assert [x.text for x in claudePolicy(LLMClient('claude-haiku-5-5', client=FakeClient(fakeReply)), 3)(reqs)] == seq
# defaults unchanged: the clue schema and task 'clue', the same request as a direct call; a schema and task can be passed
c1, c2 = LLMClient('claude-haiku-5-5', client=FakeClient(fakeReply), timing=[]), LLMClient('claude-haiku-5-5', client=FakeClient(fakeReply))
claudePolicy(c1)(reqs[:1]); c2.complete(*reqs[0], CLUE_SCHEMA, task='clue')
assert c1.client.requests == c2.client.requests == [c2.request(*reqs[0], CLUE_SCHEMA)] and c1.timing[0]['task'] == 'clue'
S = {'type': 'object', 'additionalProperties': False, 'required': ['vote', 'notes'],
     'properties': {'vote': {'type': 'string', 'enum': ['A', 'B']}, 'notes': {'type': 'string'}}}
c3 = LLMClient('claude-haiku-5-5', client=FakeClient(['{"vote": "A", "notes": ""}']), timing=[])
assert claudePolicy(c3, 2, S, 'profile')(reqs[:2])[1].parsed['vote'] == 'A'
assert c3.client.requests[0]['output_config']['format']['schema'] == S and {x['task'] for x in c3.timing} == {'profile'}

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()